# Import and Read data


In [1]:
import sys
sys.path.append('.')  # Add current directory to path
from training import *
import torch
from itertools import product


/opt/conda/lib/python3.11/site-packages/transformers/utils/hub.py:127: FutureWarning: Using `TRANSFORMERS_CACHE` is deprecated and will be removed in v5 of Transformers. Use `HF_HOME` instead.
  warnings.warn(


In [2]:
partition = 478

In [3]:
df = pd.read_csv(f"../../data/top30groups/anonLoc/combined/combined{partition}.csv")

In [4]:
from pathlib import Path
import numpy as np
from datetime import datetime

def save_metrics_txt(best_metrics, partition, best_params, append=False):
    """
    Write best_metrics to results/results_{partition}.txt as key: value lines.
    Set append=True to add another block instead of overwriting.
    """
    results_dir = Path("results")
    results_dir.mkdir(parents=True, exist_ok=True)
    path = results_dir / f"results_{partition}.txt"
    mode = "a" if append else "w"

    # make JSON-friendly scalars for numpy types
    def to_scalar(v):
        if isinstance(v, (np.floating,)):
            return float(v)
        if isinstance(v, (np.integer,)):
            return int(v)
        return v

    with open(path, mode, encoding="utf-8") as f:
        if append:
            f.write("\n" + "="*60 + "\n")
        f.write(f"Run saved: {datetime.now().isoformat(timespec='seconds')}\n")
        # align keys for readability
        f.write(f"{best_params}\n")
        width = max(len(k) for k in best_metrics.keys())
        for k in sorted(best_metrics.keys()):
            f.write(f"{k:<{width}} : {to_scalar(best_metrics[k])}\n")

# usage:
# save_metrics_txt(best_metrics, partition="gtd300", append=False)


In [5]:
import random 
node_feature_cols = [c for c in df.columns if c!='gname']
edge_feature_cols = ['attacktype1', 'target1']
edge_mode = 'hybrid_equal_knn'
equal_cols = ['encodedlonglat']
k = 8


param_grid = {
    'hidden_dims': [64],
    'dropouts_gcn': [0.3],
    'lr': [0.01],
    'n_tree': [50,70,100],
    'tree_depth': [7,8,9,10],
    'tree_feature_rate': [0.1,0.3,0.5],
    'feat_dropout': [0.1,0.3],
    'out_size_nrf': [256,512,768],
    'ks' : [8],
    'wd' : [5e-4, 5e-3,0.01]
    }

grid_combos = list(product(*param_grid.values()))
param_names = list(param_grid.keys())
sampled_combos = random.sample(grid_combos, 50)

best_acc = -1
for combo in sampled_combos:
    param_dict = dict(zip(param_names, combo))
    print(combo)
    nrf_cfg = {
        **dict(zip(param_names, combo)),
        "epochs": 300,
        "partition": f"gtd{partition}",   # one of: "gtd100", "gtd200", "gtd300", "gtd478"
        "final_evaluation": False,
        "n_class": 30
    }

    test_acc, best_epoch, best_metrics, epoch_logs = train_joint_gcn_nrf(
        df,
        node_feature_cols,
        edge_feature_cols,
        edge_mode,
        equal_cols,
        k,
        param_dict['hidden_dims'],
        param_dict['dropouts_gcn'],
        nrf_cfg,
        weight_decay=5e-4,
        device="cuda"
    )

    if test_acc > best_acc:
        best_acc = test_acc
        best_params = nrf_cfg

print(best_params)


(64, 0.3, 0.01, 50, 10, 0.1, 0.1, 256, 8, 0.005)
Best validation acc: 0.5257 @ epoch 297
(64, 0.3, 0.01, 100, 7, 0.1, 0.3, 768, 8, 0.01)
Best validation acc: 0.5201 @ epoch 286
(64, 0.3, 0.01, 50, 10, 0.3, 0.3, 512, 8, 0.01)
Best validation acc: 0.5403 @ epoch 278
(64, 0.3, 0.01, 100, 10, 0.1, 0.3, 512, 8, 0.01)
Best validation acc: 0.5264 @ epoch 294
(64, 0.3, 0.01, 70, 9, 0.5, 0.1, 256, 8, 0.005)
Best validation acc: 0.5479 @ epoch 290
(64, 0.3, 0.01, 70, 9, 0.1, 0.3, 768, 8, 0.005)
Best validation acc: 0.5312 @ epoch 295
(64, 0.3, 0.01, 50, 10, 0.1, 0.3, 256, 8, 0.0005)
Best validation acc: 0.5059 @ epoch 299
(64, 0.3, 0.01, 50, 8, 0.1, 0.1, 768, 8, 0.0005)
Best validation acc: 0.5323 @ epoch 287
(64, 0.3, 0.01, 70, 9, 0.5, 0.3, 768, 8, 0.0005)
Best validation acc: 0.5500 @ epoch 296
(64, 0.3, 0.01, 100, 9, 0.1, 0.1, 256, 8, 0.0005)
Best validation acc: 0.5250 @ epoch 299
(64, 0.3, 0.01, 100, 7, 0.5, 0.1, 768, 8, 0.005)
Best validation acc: 0.5372 @ epoch 287
(64, 0.3, 0.01, 70, 10,

In [6]:
save_metrics_txt(best_metrics, partition, best_params)